In [14]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("cwd:", Path.cwd())
print("project_root:", project_root)
print("sys.path[0]:", sys.path[0])

cwd: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models
project_root: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5
sys.path[0]: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5


In [15]:
import sys

# remover torchtools já carregado, se existir
for k in list(sys.modules.keys()):
    if k == "torchtools" or k.startswith("torchtools."):
        del sys.modules[k]

print("torchtools removed from sys.modules if it existed")

torchtools removed from sys.modules if it existed


In [16]:
import importlib

import torchtools
print("torchtools imported from:", torchtools.__file__)

2026-04-19 00:14:32.227309: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-19 00:14:32.292281: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-19 00:14:32.292362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-19 00:14:32.294040: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-19 00:14:32.303741: I tensorflow/core/platform/cpu_feature_guar

torchtools imported from: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/torchtools/__init__.py


In [17]:
from torchtools.model import ModelOutput, ModelRunner
print("Import successful")

Import successful


In [18]:
import torch
import torch.nn as nn

from hrnet.model_hrnet_pep import HRNetForPepSegmentation
from segformer_b2.model_segformer_b2_pep import SegFormerB2PepBackbone
from miml.model_miml_pep import MIMLPepBackbone

In [19]:
def count_parameters(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable
    return {
        "total": total,
        "trainable": trainable,
        "non_trainable": non_trainable,
        "total_m": total / 1e6,
    }

def pretty_print(name: str, stats: dict):
    print(f"=== {name} ===")
    print(f"Total params        : {stats['total']:,}")
    print(f"Trainable params    : {stats['trainable']:,}")
    print(f"Non-trainable params: {stats['non_trainable']:,}")
    print(f"Total params (M)    : {stats['total_m']:.2f}M")
    print()

In [20]:
hrnet = HRNetForPepSegmentation()
segformer_b2 = SegFormerB2PepBackbone(num_labels=1)
miml = MIMLPepBackbone(patch_size=4, feat_dim=360)

Fetching 6 files: 100%|████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 14944.08it/s]
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at /home/guests3/rma/.cache/hf_models/models--nvidia--mit-b2/snapshots/3bb39e8739149c3777d0325349b2a6c32c6413db and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to 

In [21]:
models = {
    "HRNet": hrnet,
    "SegFormer-B2": segformer_b2,
    "MiML": miml,
}

results = {}
for name, model in models.items():
    stats = count_parameters(model)
    results[name] = stats
    pretty_print(name, stats)

=== HRNet ===
Total params        : 70,309,777
Trainable params    : 70,309,777
Non-trainable params: 0
Total params (M)    : 70.31M

=== SegFormer-B2 ===
Total params        : 27,347,393
Trainable params    : 27,347,393
Non-trainable params: 0
Total params (M)    : 27.35M

=== MiML ===
Total params        : 71,591,529
Trainable params    : 71,591,529
Non-trainable params: 0
Total params (M)    : 71.59M



In [22]:
import pandas as pd

df = pd.DataFrame(results).T
df

,total,trainable,non_trainable,total_m
HRNet,70309777.0,70309777.0,0.0,70.309777
SegFormer-B2,27347393.0,27347393.0,0.0,27.347393
MiML,71591529.0,71591529.0,0.0,71.591529


In [23]:
for name, stats in results.items():
    print(f"{name}: {stats['total_m']:.2f}M")

HRNet: 70.31M
SegFormer-B2: 27.35M
MiML: 71.59M
